# Scoring a live extraction agent

Five FCC political-advertising invoices from
[RealKIE-FCC-Verified](https://huggingface.co/datasets/amazon-agi/RealKIE-FCC-Verified), extracted live
by Claude Haiku 4.5 and scored field by field.

The setup is the ordinary one: you already have a Pydantic model bound to the agent's structured
output, and you want to know how good the extraction is. The same model drives the comparison, so
there is nothing extra to write.

The last section shows the other supported path, binding a configured stickler `StructuredModel` as
the agent's `structured_output_model`, for when you have already tuned comparators and thresholds.

Requires AWS credentials with Bedrock access to `us.anthropic.claude-haiku-4-5`.

In [1]:
import json
import urllib.parse
import urllib.request
from typing import List, Optional

import boto3
from pydantic import BaseModel, Field
from strands import Agent
from strands.models import BedrockModel
from strands_evals import Case, Experiment, eval_task

from stickler.integrations.strands_evals import StructuredOutputEvaluator

N_DOCS = 5
MAX_LINE_ITEMS = 12          # Hungarian matching is cubic in list length
MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
OCR_CHAR_LIMIT = 12_000

## The documents

In [2]:
url = (
    "https://datasets-server.huggingface.co/rows"
    f"?dataset={urllib.parse.quote('amazon-agi/RealKIE-FCC-Verified', safe='')}"
    "&config=default&split=test&offset=0&length=25"
)
with urllib.request.urlopen(url, timeout=60) as resp:
    rows = [item["row"] for item in json.load(resp)["rows"]]

rows = [r for r in rows if len(r["json_response"].get("LineItems") or []) <= MAX_LINE_ITEMS][:N_DOCS]

print(f"{len(rows)} documents")
for r in rows:
    print(f"  {r['id'][:28]:30} {len(r['json_response'].get('LineItems') or []):>2} line items")

5 documents
  3dabc131205d3a2c74a73eb54549    4 line items
  75986ca8ed4b97420672b7782390    4 line items
  a501c2b139c99cc56e6d41a8f9df    2 line items
  64e40cb2f126bad63679f19505e8    3 line items
  08e2649c3291ee569a116260d46e    0 line items


## Your model

Plain Pydantic, no stickler annotations. Every nullable field is `Optional`, because on a scanned
invoice those fields are legitimately absent and "correctly returned nothing" has to score as a
success rather than a miss.

In [3]:
class FCCLineItem(BaseModel):
    LineItemDescription: Optional[str] = None
    LineItemStartDate: Optional[str] = None
    LineItemEndDate: Optional[str] = None
    LineItemDays: Optional[str] = None
    LineItemRate: Optional[float] = None


class FCCInvoice(BaseModel):
    Agency: str
    Advertiser: str
    GrossTotal: Optional[float] = None
    PaymentTerms: Optional[str] = None
    AgencyCommission: Optional[float] = None
    NetAmountDue: Optional[float] = None
    LineItems: List[FCCLineItem] = Field(default_factory=list)

## The task

`@eval_task()` wraps the function the harness calls per case, so the Bedrock call happens as part of
the evaluation and the harness handles concurrency.

Returning a `dict` matters: `EvalTaskHandler` passes a dict through untouched but calls `str()` on
anything else, which would flatten the structured output into text.

In [4]:
SESSION = boto3.Session(region_name="us-east-1")
SESSION.client("sts").get_caller_identity()      # free, fails fast on a stale token
BEDROCK = BedrockModel(model_id=MODEL, boto_session=SESSION)

PROMPT = (
    "Extract the invoice fields from this FCC political advertising invoice. Return null for any "
    "field not shown on the document. Transcribe dates exactly as printed. Include every row of the "
    "line-item table.\n\nDOCUMENT:\n{text}"
)


@eval_task()
def extract(case):
    agent = Agent(model=BEDROCK, system_prompt="You extract invoice data.", callback_handler=None)
    result = agent(
        PROMPT.format(text=case.input),
        structured_output_model=case.metadata["model"],
    )
    if result.structured_output is None:
        raise ValueError(f"no structured output for {case.name}")
    return {"output": result.structured_output}

## Run it

Ground truth is the dataset's own `json_response`, validated into the same model the agent fills.

In [5]:
cases = [
    Case[str, FCCInvoice](
        name=r["id"][:28],
        input=r["text"][:OCR_CHAR_LIMIT],
        expected_output=FCCInvoice.model_validate(r["json_response"]),
        metadata={"model": FCCInvoice},
    )
    for r in rows
]

evaluator = StructuredOutputEvaluator(FCCInvoice)
report = await Experiment[str, FCCInvoice](
    cases=cases, evaluators=[evaluator]
).run_evaluations_async(extract)

report.display(include_input=False)

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.82           Pass Rate: 0.4                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                        Test Case Results                                        
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ index ┃ name                         ┃ evaluator                 ┃ score ┃ test_pass ┃ reason ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ ▶ 0   │ 3dabc131205d3a2c74a73eb54549 │ StructuredOutputEvaluator │ 0.99  │ ✅        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 1   │ 75986ca8ed4b97420672b7782390 │ StructuredOutputEvaluator │ 0.51  │ ❌        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 2   │ a501c2b139c99cc56e6d41a8f9df │ StructuredOutputEvaluator │ 0.94  │ ✅        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 3   │ 64e40cb2f126bad63679f19505e8 │ StructuredOutputEvaluator │ 0.83  │ ❌        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 4   │ 08e2649c3291ee569a116260d46e │ StructuredOutputEvaluator │ 0.86  │ ❌        │ ...    │
└───────┴──────────────────────────────┴───────────────────────────┴───────┴───────────┴────────┘

Per-document scores spread across the range, which is the whole point: a single number per document
that ranks them, rather than the 0.0-or-1.0 that whole-object equality gives.

In [6]:
for entry in evaluator.per_case():
    weak = {f: round(s, 2) for f, s in entry["field_scores"].items() if s < 1.0}
    print(f"{entry['case']:30} {entry['overall_score']:>6.3f}  weak={weak or '-'}")

08e2649c3291ee569a116260d46e    0.857  weak={'Advertiser': 0.0}
3dabc131205d3a2c74a73eb54549    0.993  weak={'LineItems': 0.95}
a501c2b139c99cc56e6d41a8f9df    0.943  weak={'LineItems': 0.6}
75986ca8ed4b97420672b7782390    0.506  weak={'Agency': 0.0, 'Advertiser': 0.0, 'AgencyCommission': 0.0, 'LineItems': 0.54}
64e40cb2f126bad63679f19505e8    0.826  weak={'Advertiser': 0.98, 'PaymentTerms': 0.0, 'LineItems': 0.8}


## Which field is broken

`metrics()` returns the five-category confusion matrix per field path, including nested paths. It runs
no extra comparisons: each document was compared once above and the raw result was kept.

**FN** is a field the extractor missed, **FA** one it invented, **FD** one it got wrong. Those need
different fixes, and a single score cannot separate them.

Nested `LineItems.*` rows only count documents whose line-item pair scored at or above
`match_threshold`; below that, threshold gating emits no field breakdown, so those documents show up
as `fd` on `LineItems` instead.

In [7]:
rollup = evaluator.metrics()["FCCInvoice"]

print(f"documents: {rollup.document_count}\n")
print(f"{'field':30} {'tp':>3} {'fn':>3} {'fa':>3} {'fd':>3}  {'prec':>5} {'rec':>5} {'f1':>5}")
print("-" * 66)
for field, m in sorted(rollup.field_metrics.items(),
                       key=lambda kv: (kv[1].get("cm_f1", 1.0), kv[0])):
    print(f"{field:30} {m.get('tp', 0):>3} {m.get('fn', 0):>3} {m.get('fa', 0):>3} {m.get('fd', 0):>3}"
          f"  {m.get('cm_precision', 0):>5.2f} {m.get('cm_recall', 0):>5.2f} {m.get('cm_f1', 0):>5.2f}")

worst = min(rollup.field_metrics.items(), key=lambda kv: kv[1].get("cm_f1", 1.0))
print(f"\nWorst field: {worst[0]} (F1 {worst[1].get('cm_f1', 0):.2f}) -- the effort belongs here.")

documents: 5

field                           tp  fn  fa  fd   prec   rec    f1
------------------------------------------------------------------
LineItems                        7   0   0   6   0.54  1.00  0.70
LineItems.LineItemDays           4   0   0   3   0.57  1.00  0.73
Advertiser                       3   0   0   2   0.60  1.00  0.75
Agency                           4   0   0   1   0.80  1.00  0.89
AgencyCommission                 4   0   0   1   0.80  1.00  0.89
PaymentTerms                     4   0   1   0   0.80  1.00  0.89
GrossTotal                       5   0   0   0   1.00  1.00  1.00
LineItems.LineItemDescription    7   0   0   0   1.00  1.00  1.00
LineItems.LineItemEndDate        7   0   0   0   1.00  1.00  1.00
LineItems.LineItemRate           7   0   0   0   1.00  1.00  1.00
LineItems.LineItemStartDate      7   0   0   0   1.00  1.00  1.00
NetAmountDue                     5   0   0   0   1.00  1.00  1.00

Worst field: LineItems (F1 0.70) -- the effort belongs here.

## Why each field scored that way

Nothing was configured, so every comparator and threshold was inferred from the model's field names
and types. `explain()` shows what was chosen and on what basis, which is what makes a score
defensible.

In [8]:
print(f"{'field':30} {'comparator':40} {'thr':>5}  basis")
print("-" * 86)
for path, cfg in evaluator.explain().items():
    print(f"{path:30} {cfg['comparator']:40} {cfg['threshold']:>5}  {cfg['source']}")

field                          comparator                                 thr  basis
--------------------------------------------------------------------------------------
Agency                         LevenshteinComparator                      0.7  type
Advertiser                     LevenshteinComparator                      0.7  type
GrossTotal                     NumericComparator                         0.95  name-token
PaymentTerms                   LevenshteinComparator                      0.7  type
AgencyCommission               NumericComparator                         0.95  type
NetAmountDue                   NumericComparator                         0.95  name-token
LineItems                      Hungarian (per-element StructuredModel)    0.7  type
LineItems.LineItemDescription  FuzzyComparator                            0.6  name-token
LineItems.LineItemStartDate    LevenshteinComparator                      0.7  name-token
LineItems.LineItemEndDate      LevenshteinCompar

## Already tuned a StructuredModel?

If you have configured comparators and thresholds on a stickler `StructuredModel`, bind that as the
agent's `structured_output_model` directly. It is a `BaseModel` subclass, and its rendered JSON Schema
now describes the same shape a plain model would, so Strands sees no difference: `required` is derived
from the annotation, nothing widens to `["string", "null"]`, and the comparison config stays out of
the tool spec where it would only be prompt noise.

The dates are the clearest reason to bother. In this dataset they arrive as strings (`"10/05/16"`), so
inference sees `str` and picks `LevenshteinComparator`. Edit distance is the wrong tool for a date, and
not merely imprecise: it ranks the cases backwards.

| ground truth | prediction | Levenshtein | DateComparator | |
|---|---|---|---|---|
| `10/05/16` | `2016-10-05` | 0.100 | **1.000** | same date, ISO form |
| `10/05/16` | `10/5/16` | 0.875 | **1.000** | same date, unpadded |
| `10/05/16` | `10/06/16` | 0.875 | **0.000** | **different date** |

Levenshtein scores a *different* date higher than the same date reformatted, so no threshold separates
them. `DateComparator` parses both sides, so format stops mattering and the day starts mattering.

In [9]:
from strands.tools.structured_output import convert_pydantic_to_tool_spec

from stickler import (
    ComparableField,
    DateComparator,
    ExactComparator,
    LevenshteinComparator,
    NumericComparator,
    StructuredModel,
)


class TunedLineItem(StructuredModel):
    LineItemDescription: Optional[str] = ComparableField(comparator=LevenshteinComparator(), threshold=0.8)
    LineItemStartDate: Optional[str] = ComparableField(comparator=DateComparator())
    LineItemEndDate: Optional[str] = ComparableField(comparator=DateComparator())
    LineItemDays: Optional[str] = ComparableField(comparator=ExactComparator())
    LineItemRate: Optional[float] = ComparableField(comparator=NumericComparator(), threshold=0.99)


class TunedInvoice(StructuredModel):
    Agency: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.9, weight=2.0)
    Advertiser: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.9, weight=2.0)
    GrossTotal: Optional[float] = ComparableField(comparator=NumericComparator(), threshold=0.99, weight=3.0)
    PaymentTerms: Optional[str] = ComparableField(comparator=LevenshteinComparator(), threshold=0.7)
    AgencyCommission: Optional[float] = ComparableField(comparator=NumericComparator(), threshold=0.99)
    NetAmountDue: Optional[float] = ComparableField(comparator=NumericComparator(), threshold=0.99, weight=3.0)
    LineItems: List[TunedLineItem] = ComparableField(weight=2.0)

    match_threshold = 0.8


spec = convert_pydantic_to_tool_spec(TunedInvoice)["inputSchema"]["json"]
blob = json.dumps(spec)
print(f"required        : {spec.get('required')}")
print(f"Agency type     : {spec['properties']['Agency'].get('type')}")
print(f"config leaked   : {'x-comparison' in blob or 'x-aws-stickler' in blob}")
print(f"extra_fields    : {'extra_fields' in blob}")

required        : ['Agency', 'Advertiser', 'LineItems']
Agency type     : string
config leaked   : False
extra_fields    : False


The tool spec is clean, so the agent is given the same instructions it would get from a plain model.
Scoring it uses the explicit configuration instead of inference.

In [10]:
tuned_cases = [
    Case[str, TunedInvoice](
        name=r["id"][:28],
        input=r["text"][:OCR_CHAR_LIMIT],
        expected_output=TunedInvoice.from_json(r["json_response"]),
        metadata={"model": TunedInvoice},
    )
    for r in rows
]

tuned_eval = StructuredOutputEvaluator(TunedInvoice)
tuned_report = await Experiment[str, TunedInvoice](
    cases=tuned_cases, evaluators=[tuned_eval]
).run_evaluations_async(extract)

tuned_report.display(include_input=False)

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.79           Pass Rate: 0.2                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                        Test Case Results                                        
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ index ┃ name                         ┃ evaluator                 ┃ score ┃ test_pass ┃ reason ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ ▶ 0   │ 3dabc131205d3a2c74a73eb54549 │ StructuredOutputEvaluator │ 0.98  │ ✅        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 1   │ 75986ca8ed4b97420672b7782390 │ StructuredOutputEvaluator │ 0.35  │ ❌        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 2   │ a501c2b139c99cc56e6d41a8f9df │ StructuredOutputEvaluator │ 0.91  │ ❌        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 3   │ 64e40cb2f126bad63679f19505e8 │ StructuredOutputEvaluator │ 0.87  │ ❌        │ ...    │
├───────┼──────────────────────────────┼───────────────────────────┼───────┼───────────┼────────┤
│ ▶ 4   │ 08e2649c3291ee569a116260d46e │ StructuredOutputEvaluator │ 0.86  │ ❌        │ ...    │
└───────┴──────────────────────────────┴───────────────────────────┴───────┴───────────┴────────┘

The same evaluation as the inferred run, on the same five documents. Below, the per-field rollup and
the configuration that produced it.

In [11]:
tuned_rollup = tuned_eval.metrics()["TunedInvoice"]

print(f"documents: {tuned_rollup.document_count}\n")
print(f"{'field':30} {'tp':>3} {'fn':>3} {'fa':>3} {'fd':>3}  {'prec':>5} {'rec':>5} {'f1':>5}")
print("-" * 66)
for path, m in sorted(tuned_rollup.field_metrics.items(),
                      key=lambda kv: (kv[1].get("cm_f1", 1.0), kv[0])):
    print(f"{path:30} {m.get('tp', 0):>3} {m.get('fn', 0):>3} {m.get('fa', 0):>3} {m.get('fd', 0):>3}"
          f"  {m.get('cm_precision', 0):>5.2f} {m.get('cm_recall', 0):>5.2f} {m.get('cm_f1', 0):>5.2f}")

print(f"\n{'field':22} {'comparator':30} {'thr':>5}  basis")
print("-" * 68)
for path, cfg in tuned_eval.explain().items():
    print(f"{path:22} {cfg['comparator']:30} {cfg['threshold']:>5}  {cfg['source']}")

documents: 5

field                           tp  fn  fa  fd   prec   rec    f1
------------------------------------------------------------------
LineItems.LineItemDescription    2   0   0   5   0.29  1.00  0.44
LineItems.LineItemDays           3   0   0   4   0.43  1.00  0.60
LineItems                        7   0   3   6   0.44  1.00  0.61
Advertiser                       4   0   0   1   0.80  1.00  0.89
Agency                           4   0   0   1   0.80  1.00  0.89
AgencyCommission                 4   0   0   1   0.80  1.00  0.89
NetAmountDue                     4   0   0   1   0.80  1.00  0.89
PaymentTerms                     4   0   1   0   0.80  1.00  0.89
GrossTotal                       5   0   0   0   1.00  1.00  1.00
LineItems.LineItemEndDate        7   0   0   0   1.00  1.00  1.00
LineItems.LineItemRate           7   0   0   0   1.00  1.00  1.00
LineItems.LineItemStartDate      7   0   0   0   1.00  1.00  1.00

field                  comparator                       thr 

`basis` reads `explicit` for every field, where the inferred run said `name-token` or `type`. Same
harness, same agent, but the thresholds, weights and comparators are yours rather than guessed from
field names. Both date fields score F1 `1.00` with no false discoveries, where inference had them on
edit distance.

Two things to know about configuring explicitly:

**You have to name the comparator.** A `float` declared as `ComparableField(threshold=0.99)` gets the
default string comparator, not `NumericComparator`. Inference picks the type-appropriate one for you;
explicit means explicit.

**`explain()` lists only top-level fields here**, which is why the line-item comparators are absent
from the table above and `metrics()` is used instead. `LineItems` also reports
`LevenshteinComparator`, which is wrong: a `List[StructuredModel]` is matched element-by-element and
compared recursively, so it has no single comparator.
[#250](https://github.com/awslabs/stickler/pull/250) corrects that label.